# Topic 8a: Linear Regression, MSE, and Gradient Descent

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 8a lecture note](https://dongchul.kim/csci3320-fall2026/topic/8), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## The phenomenon: six students

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5, 6])        # hours studied
y = np.array([7, 20, 38, 50, 55, 76])   # exam score

for hours, score in zip(x, y):
    print(f"{hours} hours studied  ->  {score} points")

## Look at it first

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

plt.scatter(x, y, color="black", zorder=3)
plt.xlabel("hours studied")
plt.ylabel("exam score")
plt.title("Six students")
plt.grid(alpha=0.3)
plt.show()

## A model for the phenomenon

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

grid = np.linspace(0, 7, 100)
plt.scatter(x, y, color="black", zorder=3, label="the data")
for w, b in [(0, 41), (8, 10), (16, -8), (12, 0)]:
    plt.plot(grid, w * grid + b, label=f"w={w}, b={b}")

plt.xlabel("hours studied")
plt.ylabel("exam score")
plt.title("Four of the infinitely many linear models")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

## What "best" means: error

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

flat = 0 * x + 41          # a flat line at the average score. Obviously terrible.
residuals = flat - y

print("predictions:", flat)
print("residuals:  ", residuals)
print("added up:   ", residuals.sum())          # zero!
print("cost J:     ", (residuals ** 2).mean() / 2)

## One knob, to see the whole picture

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

def cost(w):
    y_pred = w * x
    return ((y_pred - y) ** 2).mean() / 2      # the 1/2n cost

for w in [5, 12, 20]:
    print(f"w = {w:2}  ->  predictions {w * x}  ->  J = {cost(w):8.2f}")
print(f"       actual scores {y}")

## The cost is a bowl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

def cost(w):
    return ((w * x - y) ** 2).mean() / 2

ws = np.linspace(0, 24, 200)                 # 200 candidate rates
costs = [cost(w) for w in ws]

plt.plot(ws, costs)
plt.axvline(12, color="green", linestyle="--", label="w = 12")
plt.xlabel("w  (points per hour of study)")
plt.ylabel("J")
plt.title("The cost as a function of the knob")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## The training loop

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

w = 0.0          # start anywhere
alpha = 0.02     # the learning rate
n = len(x)

print(" iter        w            J")
for i in range(61):
    y_pred = w * x                          # 1. predict
    cost = ((y_pred - y) ** 2).mean() / 2   # 2. measure how wrong
    grad = ((y_pred - y) * x).mean()        # 3. which way is downhill
    if i % 10 == 0:
        print(f"{i:5}  {w:8.4f}  {cost:11.4f}")
    w = w - alpha * grad                    # 4. step downhill

print(f"\nlearned rate: {w:.2f} points per hour of study")

## The learning curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

def train(alpha, steps=60):
    w, history = 0.0, []
    for _ in range(steps):
        y_pred = w * x
        history.append(((y_pred - y) ** 2).mean() / 2)
        w = w - alpha * ((y_pred - y) * x).mean()
    return history

plt.plot(train(0.002), label="alpha = 0.002  (too small)")
plt.plot(train(0.020), label="alpha = 0.02   (good)")
plt.plot(train(0.140), label="alpha = 0.14   (too big)")
plt.xlabel("iteration")
plt.ylabel("J")
plt.title("The learning curve")
plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## scikit-learn does all of it

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

X = x.reshape(-1, 1)                            # scikit-learn wants a 2-D X

model = LinearRegression(fit_intercept=False)   # our model: y = w*x, no intercept
model.fit(X, y)                                 # the entire training loop

print(f"scikit-learn w : {model.coef_[0]:.4f}")
print(f"our loop    w : 12.0000")
print(f"predicted score after 7 hours: {model.predict([[7]])[0]:.1f}")

## The second knob: updating b as well

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])

w, b = 0.0, 0.0
alpha = 0.06
n = len(x)

print(" iter        w          b            J")
for i in range(1001):
    y_pred = w * x + b                        # 1. predict, now with b
    cost   = ((y_pred - y) ** 2).mean() / 2   # 2. measure how wrong
    grad_w = ((y_pred - y) * x).mean()        # 3a. downhill for w
    grad_b = (y_pred - y).mean()              # 3b. downhill for b
    if i % 200 == 0:
        print(f"{i:5}  {w:8.4f}  {b:9.4f}  {cost:10.4f}")
    w = w - alpha * grad_w                    # 4a. step w
    b = b - alpha * grad_b                    # 4b. step b

print(f"\nlearned line: score = {w:.1f} x hours + ({b:.1f})")
print(f"final J     : {((w * x + b - y) ** 2).mean() / 2:.2f}   (one knob managed 7.50)")

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([7, 20, 38, 50, 55, 76])
X = x.reshape(-1, 1)

model = LinearRegression()        # fit_intercept=True is the default
model.fit(X, y)

print(f"scikit-learn : w = {model.coef_[0]:.4f}, b = {model.intercept_:.4f}")
print(f"our loop     : w = 13.2000, b = -5.2000")

Back to the [Topic 8a lecture note](https://dongchul.kim/csci3320-fall2026/topic/8).